# Despliegue de la demo con Aya-101 en Google Colab

Este notebook levanta la interfaz de Streamlit del trabajo final en una GPU T4 de Colab y la abre con un enlace público.

Como el repo es privado, abre este notebook desde Colab: **Archivo → Abrir notebook → GitHub**, marca **Incluir repositorios privados** y busca `joan999/aya101_historias_clinicas`. También puedes descargar el .ipynb y subirlo con **Archivo → Subir notebook**.

Antes de empezar, en el menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, elige **T4 GPU**.

Pasos:
1. Revisar la GPU.
2. Clonar el repositorio.
3. Instalar las librerías.
4. Descargar Aya-101 (51,7 GB en fp32; tarda varios minutos).
5. Iniciar Streamlit y abrir el enlace.

El modelo se carga en 4 bits dentro de Streamlit la primera vez que se abre la página. Esa carga también tarda unos minutos.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repositorio

El repo es privado, así que Colab necesita un token de GitHub de solo lectura. Se configura una sola vez:

1. En GitHub: **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**.
   - *Repository access*: **Only select repositories** → `aya101_historias_clinicas`.
   - *Permissions*: **Contents → Read-only**.
2. En Colab: ícono de llave (**Secrets**) en la barra izquierda → **Add new secret**, con nombre `GITHUB_TOKEN` y el token como valor. Activa **Notebook access**.

Si no hay token, la celda intenta clonar sin él (sirve si el repo es público). `RAMA` elige qué rama se clona. Si la carpeta ya existe, trae los últimos cambios de esa rama. El token no se imprime ni queda guardado en la carpeta clonada.

In [ ]:
REPO_URL = "https://github.com/joan999/aya101_historias_clinicas.git"
RAMA = "main"   # rama que se clona

import os
import subprocess


def token_github():
    """Lee el token de los Secrets de Colab. Si no está, devuelve None."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception as error:
        print("Sin token de GitHub:", type(error).__name__)
        return None


def git(*args, token=None, cwd=None):
    """Corre git y tapa el token en lo que imprime."""
    resultado = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True,
                               env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    texto = (resultado.stdout + resultado.stderr).strip()
    if token:
        texto = texto.replace(token, "***")
    if texto:
        print(texto)
    if resultado.returncode != 0:
        raise RuntimeError("git falló: revisa el secret GITHUB_TOKEN y que el token tenga acceso al repo")


token = token_github()
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
carpeta = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if os.path.exists(carpeta):
    git("pull", "-q", url, RAMA, token=token, cwd=carpeta)
else:
    git("clone", "-q", "--branch", RAMA, url, carpeta, token=token)
    # Deja la URL sin token en la configuración del clon
    git("remote", "set-url", "origin", REPO_URL, cwd=carpeta)
print("Código listo en", carpeta)
%cd {carpeta}
!ls

## 3. Instalar librerías

Son las mismas versiones del `uv.lock` del proyecto. Colab ya trae PyTorch con CUDA, así que no se reinstala.

In [ ]:
!pip install -q transformers==5.24.0 accelerate==1.16.0 bitsandbytes==0.50.2 streamlit==1.67.0 pymupdf==2.1

## 4. Descargar Aya-101

Se descarga una sola vez por sesión de Colab. Primero revisa que haya más de 55 GB libres en el disco.

In [ ]:
!df -h /content
from huggingface_hub import snapshot_download
ruta = snapshot_download("CohereLabs/aya-101")
print("Modelo en:", ruta)

## 5. Iniciar Streamlit y abrir el enlace

Streamlit corre en segundo plano y `cloudflared` crea un enlace público temporal (`trycloudflare.com`), sin cuenta. Al abrir el enlace, espera a que termine de cargar el modelo.

In [ ]:
import re, time

!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
!nohup /content/cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /content/cloudflared.log 2>&1 &

enlace = None
for _ in range(30):
    time.sleep(2)
    hallado = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if hallado:
        enlace = hallado.group()
        break
print("Abre la app en:", enlace or "no apareció el enlace; revisa /content/cloudflared.log")

## 6. Registro de Streamlit

Si la página muestra un error, ejecuta esta celda y copia el final del registro.

In [ ]:
!tail -n 40 /content/streamlit.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv